# 4.5 Basis and Dimension

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 4 — Vector Spaces**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import multiply
from linalg_core.elimination import solve, to_rref
from linalg_core.vecspace import is_independent, is_in_span, rank, column_space, row_space
from linalg_core import EPSILON


def fmt_vec(v):
    """Pretty-print a vector as a bracketed row."""
    return "[" + ", ".join(f"{x:g}" for x in v) + "]"


def vectors_equal(u, v):
    """Check if two vectors are equal within EPSILON tolerance."""
    return len(u) == len(v) and all(abs(a - b) < EPSILON for a, b in zip(u, v))


print("Setup complete.")

---

## 2. Motivation

A **basis** is the Goldilocks set — **independent** (no redundancy) **AND** **spanning** (complete).

We have already seen what it means for vectors to be linearly independent (no vector is
a linear combination of the others) and what it means for a set to span a space (every
vector in the space can be written as a linear combination of the set). A basis is a set
that does *both* simultaneously.

How many vectors does a basis contain? For $\mathbb{R}^n$, the answer is exactly $n$.
This number — the **dimension** — is an intrinsic property of the space, independent of
which basis you choose. Every basis for the same space has the same size.

This is one of the deepest structural results in linear algebra: the dimension of a vector
space is well-defined. It links geometry (how many independent directions exist) to
algebra (how many unknowns minus how many constraints).

---

## 3. Build

We develop the theory of basis and dimension step by step, from the definition
through the standard basis, non-standard bases, dimension, and the key theorems
that tie everything together.

### 3.1 Definition: Basis

A set $B = \{\mathbf{v}_1, \mathbf{v}_2, \ldots, \mathbf{v}_k\}$ is a **basis** for a vector
space $V$ if it satisfies two conditions:

1. **Linear independence:** $B$ is linearly independent.
2. **Spanning:** $B$ spans $V$, i.e., every $\mathbf{v} \in V$ can be written as
   $\mathbf{v} = c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 + \cdots + c_k \mathbf{v}_k$.

**Unique representation theorem (Larson, Theorem 4.12):** If $B$ is a basis for $V$,
then every vector $\mathbf{v} \in V$ can be written as a linear combination of the basis
vectors in **exactly one way**. The coefficients $c_1, \ldots, c_k$ are unique.

*Proof sketch:* Suppose $\mathbf{v} = c_1 \mathbf{v}_1 + \cdots + c_k \mathbf{v}_k
= d_1 \mathbf{v}_1 + \cdots + d_k \mathbf{v}_k$. Subtracting:
$(c_1 - d_1)\mathbf{v}_1 + \cdots + (c_k - d_k)\mathbf{v}_k = \mathbf{0}$.
By independence, each $c_i - d_i = 0$, so $c_i = d_i$. $\square$

In [ ]:
v1 = [1, 0]
v2 = [0, 1]
B = [v1, v2]

indep, dep = is_independent(B)
print("Candidate basis B = {[1,0], [0,1]}")
print(f"  (a) Independent? {indep}")

test_vectors = [[3, 5], [-2, 7], [0, 0], [100, -42]]
all_in_span = True
for v in test_vectors:
    in_span, coeffs = is_in_span(v, B)
    if not in_span:
        all_in_span = False
    print(f"  (b) {fmt_vec(v)} in span(B)? {in_span}  coeffs = {fmt_vec(coeffs) if coeffs else 'N/A'}")

print(f"\nB is a basis for R^2: independent={indep}, spans R^2={all_in_span}")

### 3.2 Standard Basis for $\mathbb{R}^n$

The simplest basis for $\mathbb{R}^n$ is the **standard basis** (or **natural basis**):

$$\mathcal{E} = \{\mathbf{e}_1, \mathbf{e}_2, \ldots, \mathbf{e}_n\}$$

where $\mathbf{e}_i$ is the vector with a $1$ in position $i$ and $0$s elsewhere:

$$\mathbf{e}_1 = \begin{bmatrix} 1 \\ 0 \\ \vdots \\ 0 \end{bmatrix}, \quad
  \mathbf{e}_2 = \begin{bmatrix} 0 \\ 1 \\ \vdots \\ 0 \end{bmatrix}, \quad \ldots, \quad
  \mathbf{e}_n = \begin{bmatrix} 0 \\ 0 \\ \vdots \\ 1 \end{bmatrix}.$$

**Why it is a basis:**
- **Independent:** The $n \times n$ identity matrix has rank $n$, so its columns are independent.
- **Spanning:** Any $\mathbf{v} = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_n \end{bmatrix}
  = v_1 \mathbf{e}_1 + v_2 \mathbf{e}_2 + \cdots + v_n \mathbf{e}_n$.

Let’s verify this computationally for $\mathbb{R}^3$.

In [ ]:
def standard_basis(n):
    """Return the standard basis for R^n as a list of lists."""
    basis = []
    for i in range(n):
        e = [0.0] * n
        e[i] = 1.0
        basis.append(e)
    return basis


n = 3
E = standard_basis(n)
print(f"Standard basis for R^{n}:")
for i, e in enumerate(E):
    print(f"  e{i+1} = {fmt_vec(e)}")

indep, _ = is_independent(E)
print(f"\nIndependent? {indep}")

I = Matrix.identity(n)
r = rank(I)
print(f"Rank of I_{n} = {r} (equals n={n}, confirming independence)")

v = [7, -3, 11]
in_span, coeffs = is_in_span(v, E)
print(f"\n{fmt_vec(v)} in span(E)? {in_span}")
print(f"Coefficients: {fmt_vec(coeffs)}")
print(f"Reconstruction: {coeffs[0]}*e1 + {coeffs[1]}*e2 + {coeffs[2]}*e3")
print(f"  = {fmt_vec(v)}  (the coefficients are just the components!)")

### 3.3 Non-Standard Basis — Change of Representation

The standard basis is convenient but not special. Many other sets also form bases
for $\mathbb{R}^n$. Different bases reveal different structure.

**Example:** Consider $B = \left\{\begin{bmatrix}1\\1\end{bmatrix},
\begin{bmatrix}1\\-1\end{bmatrix}\right\}$ for $\mathbb{R}^2$.

To express $\mathbf{v} = \begin{bmatrix}3\\5\end{bmatrix}$ in this basis, we solve:

$$c_1 \begin{bmatrix}1\\1\end{bmatrix} + c_2 \begin{bmatrix}1\\-1\end{bmatrix}
  = \begin{bmatrix}3\\5\end{bmatrix}
  \quad\Longleftrightarrow\quad
  \begin{bmatrix}1 & 1\\1 & -1\end{bmatrix}
  \begin{bmatrix}c_1\\c_2\end{bmatrix}
  = \begin{bmatrix}3\\5\end{bmatrix}.$$

We expect a **unique** solution (because $B$ is a basis), and the solution gives us
the **coordinate vector** $[\mathbf{v}]_B = \begin{bmatrix}c_1\\c_2\end{bmatrix}$.

In [ ]:
b1 = [1, 1]
b2 = [1, -1]
B = [b1, b2]

indep, _ = is_independent(B)
print(f"B = {{[1,1], [1,-1]}}")
print(f"Independent? {indep}")

v = [3, 5]
print(f"\nExpress v = {fmt_vec(v)} in basis B:")
print(f"Solve c1*[1,1] + c2*[1,-1] = [3,5]")

aug = Matrix.from_lists([
    [1, 1, 3],
    [1, -1, 5]
])
print(f"\nAugmented matrix [B | v]:")
print(aug)

sol_type, coeffs = solve(aug)
print(f"\nSolution type: {sol_type}")
print(f"Coefficients: c1 = {coeffs[0]:g}, c2 = {coeffs[1]:g}")

recon = [coeffs[0]*b1[i] + coeffs[1]*b2[i] for i in range(2)]
print(f"\nVerification: {coeffs[0]:g}*[1,1] + {coeffs[1]:g}*[1,-1] = {fmt_vec(recon)}")
print(f"Matches v = {fmt_vec(v)}? {vectors_equal(recon, v)}")

print(f"\nCoordinate vector [v]_B = [{coeffs[0]:g}, {coeffs[1]:g}]")
print("The same geometric vector has different coordinates in different bases.")

### 3.4 Dimension

**Theorem (Larson, Theorem 4.13):** If a vector space $V$ has a basis with $n$ vectors,
then **every** basis for $V$ has exactly $n$ vectors.

This invariant $n$ is called the **dimension** of $V$, written $\dim(V)$.

Key facts:
- $\dim(\mathbb{R}^n) = n$.
- $\dim(\{\mathbf{0}\}) = 0$ (the trivial space has the empty basis).
- $\dim(P_n) = n + 1$ for the space of polynomials of degree $\leq n$.
- $\dim(M_{m \times n}) = mn$ for the space of $m \times n$ matrices.

The dimension tells us the **number of degrees of freedom** in the space.

Let’s verify that two different bases for $\mathbb{R}^3$ both have exactly 3 vectors.

In [ ]:
B1 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
B2 = [[1, 1, 0], [0, 1, 1], [1, 0, 1]]

indep1, _ = is_independent(B1)
indep2, _ = is_independent(B2)

print("Basis B1 (standard):")
for v in B1:
    print(f"  {fmt_vec(v)}")
print(f"Independent? {indep1}  |  Size: {len(B1)}")

print(f"\nBasis B2 (non-standard):")
for v in B2:
    print(f"  {fmt_vec(v)}")
print(f"Independent? {indep2}  |  Size: {len(B2)}")

A2 = Matrix.from_lists(B2)
r2 = rank(A2)
print(f"\nRank of matrix with B2 as rows: {r2}")
print(f"B2 spans R^3? {r2 == 3}")

print(f"\nBoth bases have exactly {len(B1)} = {len(B2)} vectors.")
print(f"dim(R^3) = 3.  This is an invariant of the space, not the basis.")

### 3.5 Finding a Basis from a Spanning Set

Given a set of vectors that spans a subspace $W$, we can extract a basis by:

1. Form a matrix $A$ with the vectors as **columns**.
2. Reduce $A$ to RREF.
3. The **pivot columns** of the RREF identify which *original* columns of $A$ form a basis.

The pivot columns of $A$ (not the RREF!) form a basis for $\text{Col}(A)$.
The number of pivot columns equals $\text{rank}(A) = \dim(\text{Col}(A))$.

**Example:** Find a basis for $W = \text{span}\{\mathbf{v}_1, \mathbf{v}_2, \mathbf{v}_3, \mathbf{v}_4\}$ where:

$$\mathbf{v}_1 = \begin{bmatrix}1\\2\\3\end{bmatrix}, \quad
  \mathbf{v}_2 = \begin{bmatrix}2\\4\\6\end{bmatrix}, \quad
  \mathbf{v}_3 = \begin{bmatrix}0\\1\\1\end{bmatrix}, \quad
  \mathbf{v}_4 = \begin{bmatrix}1\\3\\4\end{bmatrix}.$$

Note: $\mathbf{v}_2 = 2\mathbf{v}_1$ and $\mathbf{v}_4 = \mathbf{v}_1 + \mathbf{v}_3$,
so we expect redundancy.

In [ ]:
v1, v2, v3, v4 = [1, 2, 3], [2, 4, 6], [0, 1, 1], [1, 3, 4]
vectors = [v1, v2, v3, v4]

A = Matrix(3, 4)
for j, v in enumerate(vectors):
    for i in range(3):
        A[i, j] = v[i]

print("A (vectors as columns):")
print(A)

R = A.copy()
pivot_positions = to_rref(R)
print(f"\nRREF of A:")
print(R)

pivot_cols = sorted(col for _, col in pivot_positions)
print(f"\nPivot positions: {pivot_positions}")
print(f"Pivot columns: {pivot_cols}")

print(f"\nBasis for W = Col(A):")
basis = [vectors[j] for j in pivot_cols]
for j in pivot_cols:
    print(f"  v{j+1} = {fmt_vec(vectors[j])}")

print(f"\ndim(W) = {len(pivot_cols)}")
print(f"Rank of A = {rank(A)}")

indep_check, _ = is_independent(basis)
print(f"\nVerify basis is independent: {indep_check}")
print("\nNote: v2 = 2*v1 (redundant), v4 = v1 + v3 (redundant).")
print("RREF correctly identified v1 and v3 as the essential (pivot) columns.")

### 3.6 Extending an Independent Set to a Basis

If $S = \{\mathbf{v}_1, \ldots, \mathbf{v}_k\}$ is linearly independent in $\mathbb{R}^n$
but $k < n$, then $S$ does not span $\mathbb{R}^n$. We can **extend** $S$ to a basis
by adding vectors until we reach $n$ independent vectors.

**Algorithm:** Start with the $k$ vectors of $S$. Adjoin standard basis vectors
$\mathbf{e}_1, \mathbf{e}_2, \ldots, \mathbf{e}_n$ one at a time. After each addition,
test independence: if the enlarged set is still independent, keep the new vector;
otherwise discard it. Stop when we have $n$ vectors.

**Example:** Extend $S = \left\{\begin{bmatrix}1\\1\\0\end{bmatrix}\right\}$
to a basis for $\mathbb{R}^3$.

In [ ]:
S = [[1, 1, 0]]
n = 3

print(f"Starting independent set S:")
for v in S:
    print(f"  {fmt_vec(v)}")
print(f"Need {n} vectors for a basis of R^{n}. Currently have {len(S)}.\n")

candidates = standard_basis(n)
current = list(S)

for e in candidates:
    trial = current + [e]
    indep, _ = is_independent(trial)
    if indep:
        current.append(e)
        print(f"  Added {fmt_vec(e)} -> independent ({len(current)} vectors)")
    else:
        print(f"  Tried {fmt_vec(e)} -> dependent, skip")
    if len(current) == n:
        break

print(f"\nExtended basis for R^{n}:")
for v in current:
    print(f"  {fmt_vec(v)}")

indep_final, _ = is_independent(current)
print(f"\nIndependent? {indep_final}")
print(f"Size = {len(current)} = dim(R^{n})? {len(current) == n}")

M = Matrix(n, n)
for j, v in enumerate(current):
    for i in range(n):
        M[i, j] = v[i]
print(f"Rank = {rank(M)} (should be {n} for a spanning set)")

### 3.7 Dimension Theorems

If $\dim(V) = n$, these three results follow immediately from the theory of basis
(Larson, Theorem 4.14):

| Theorem | Statement |
|---|---|
| **(a)** Too many | If a set in $V$ has more than $n$ vectors, it is **linearly dependent**. |
| **(b)** Too few | If a set in $V$ has fewer than $n$ vectors, it **does not span** $V$. |
| **(c)** Just right | If a set of exactly $n$ vectors is linearly independent, it is automatically a **basis** for $V$. |

These theorems are powerful because they let you check **one** condition instead of two:
- An independent set of the right size must also span.
- A spanning set of the right size must also be independent.

Let’s verify all three with concrete examples.

In [ ]:
print("=" * 60)
print("(a) Too many vectors => dependent")
print("=" * 60)
print("dim(R^2) = 2, so any 3 vectors in R^2 must be dependent.\n")

w1, w2, w3 = [1, 0], [0, 1], [2, 3]
indep, dep = is_independent([w1, w2, w3])
print(f"Vectors: {fmt_vec(w1)}, {fmt_vec(w2)}, {fmt_vec(w3)}")
print(f"Independent? {indep}")
print(f"Dependency relation: {fmt_vec(dep)}")
print(f"  => {dep[0]:g}*w1 + {dep[1]:g}*w2 + {dep[2]:g}*w3 = 0")

check = [dep[0]*w1[i] + dep[1]*w2[i] + dep[2]*w3[i] for i in range(2)]
print(f"  Verify: {fmt_vec(check)}\n")

print("=" * 60)
print("(b) Too few vectors => don't span")
print("=" * 60)
print("dim(R^3) = 3, so any 2 vectors cannot span R^3.\n")

u1, u2 = [1, 0, 0], [0, 1, 0]
test_v = [0, 0, 1]
in_span, _ = is_in_span(test_v, [u1, u2])
print(f"Vectors: {fmt_vec(u1)}, {fmt_vec(u2)}")
print(f"{fmt_vec(test_v)} in span? {in_span}")
print(f"Two vectors cannot span R^3 (we need at least 3).\n")

print("=" * 60)
print("(c) Exactly n independent vectors => basis")
print("=" * 60)
print("dim(R^3) = 3, so 3 independent vectors MUST be a basis.\n")

q1, q2, q3 = [1, 2, 0], [0, 1, 1], [1, 0, -1]
Q = [q1, q2, q3]
indep, _ = is_independent(Q)
print(f"Vectors: {fmt_vec(q1)}, {fmt_vec(q2)}, {fmt_vec(q3)}")
print(f"Independent? {indep}")
print(f"Count = {len(Q)} = dim(R^3) = 3")
print(f"Therefore these vectors form a basis for R^3 (no need to check spanning separately).")

target = [5, 7, -2]
in_span, coeffs = is_in_span(target, Q)
print(f"\nDouble-check spanning: {fmt_vec(target)} in span? {in_span}")
print(f"Coefficients: {fmt_vec(coeffs)}")
recon = [sum(coeffs[j]*Q[j][i] for j in range(3)) for i in range(3)]
print(f"Reconstruction: {fmt_vec(recon)}")
print(f"Match? {vectors_equal(recon, target)}")

---

## 4. Exercises

Test your understanding with the following problems. Write code in the provided cells.

### Exercise 1 (Easy)

Verify that the standard basis $\{\mathbf{e}_1, \mathbf{e}_2, \mathbf{e}_3\}$ for
$\mathbb{R}^3$ is indeed a basis.

1. Check that it is linearly independent.
2. Check that it spans $\mathbb{R}^3$ by showing that the vector
   $\mathbf{v} = \begin{bmatrix}4\\-2\\7\end{bmatrix}$ is in the span.
3. Print the coordinate representation $\mathbf{v} = 4\mathbf{e}_1 - 2\mathbf{e}_2 + 7\mathbf{e}_3$.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Given the four vectors in $\mathbb{R}^4$:

$$\mathbf{v}_1 = \begin{bmatrix}1\\0\\1\\0\end{bmatrix}, \quad
  \mathbf{v}_2 = \begin{bmatrix}0\\1\\0\\1\end{bmatrix}, \quad
  \mathbf{v}_3 = \begin{bmatrix}1\\1\\1\\1\end{bmatrix}, \quad
  \mathbf{v}_4 = \begin{bmatrix}2\\1\\2\\1\end{bmatrix}.$$

1. Build the matrix $A$ with these vectors as columns.
2. Find the RREF and identify pivot columns.
3. Extract a basis for $W = \text{span}\{\mathbf{v}_1, \mathbf{v}_2, \mathbf{v}_3, \mathbf{v}_4\}$.
4. What is $\dim(W)$?

*Hint:* Note that $\mathbf{v}_3 = \mathbf{v}_1 + \mathbf{v}_2$ and
$\mathbf{v}_4 = 2\mathbf{v}_1 + \mathbf{v}_2$.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Write two functions:

1. `find_basis(vectors)` — Given a list of vectors (which may be dependent), return a
   subset that forms a basis for their span. Use the RREF pivot-column approach.

2. `extend_to_basis(vectors, n)` — Given an independent set of vectors in $\mathbb{R}^n$,
   extend it to a full basis for $\mathbb{R}^n$ by adding standard basis vectors as needed.

Test `find_basis` with the vectors from Exercise 2. Test `extend_to_basis` with
$S = \{[1, 0, 0, 1], [0, 1, 1, 0]\}$ in $\mathbb{R}^4$.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Definition / Key Fact |
|---|---|
| **Basis** | A set that is both linearly independent and spanning |
| **Unique representation** | Every $\mathbf{v} \in V$ has exactly one expression as a linear combination of basis vectors |
| **Standard basis** | $\{\mathbf{e}_1, \ldots, \mathbf{e}_n\}$ for $\mathbb{R}^n$; the columns of $I_n$ |
| **Dimension** | $\dim(V) = $ number of vectors in any basis for $V$; an invariant of the space |
| **Finding a basis** | RREF the column matrix $\to$ pivot columns of the *original* matrix form a basis |
| **Extending to a basis** | Adjoin standard basis vectors one at a time, keeping independence |
| **Too many ($> n$)** | Any set of more than $\dim(V)$ vectors is dependent |
| **Too few ($< n$)** | Any set of fewer than $\dim(V)$ vectors cannot span $V$ |
| **Just right ($= n$)** | $n$ independent vectors in an $n$-dimensional space $\Rightarrow$ automatically a basis |

**Takeaway:** A basis is the minimal complete description of a vector space. The dimension
is the single number that captures the “size” of the space. Once you know the dimension,
you only need to verify **one** of the two basis conditions — the other follows for free
when the set has exactly the right number of vectors. Next, in Section 4.6, we’ll use
these ideas to define **rank** and **nullity** and connect them through the
Rank–Nullity Theorem.